In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv('p6_spam.csv')
cv = CountVectorizer()
X = cv.fit_transform(df.text).toarray().astype(float)
y = (df.label == 'spam').astype(int).values
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

class MyGNB:
    def fit(self, X, y):
        self.c = np.unique(y)
        self.m = np.array([X[y == k].mean(0) for k in self.c])
        self.v = np.array([X[y == k].var(0) for k in self.c]) + 1e-9 * X.var(0).max()
        self.p = np.array([(y == k).mean() for k in self.c])
        return self

    def predict(self, X):
        ll = [np.log(p) - 0.5 * np.sum(np.log(2 * np.pi * v) + (X - m) ** 2 / v, axis=1) for p, m, v in zip(self.p, self.m, self.v)]
        return self.c[np.argmax(ll, axis=0)]

In [ ]:
a = MyGNB().fit(Xtr, ytr)
b = GaussianNB().fit(Xtr, ytr)
print('Manual  accuracy: %.3f' % accuracy_score(yte, a.predict(Xte)))
print('sklearn accuracy: %.3f' % accuracy_score(yte, b.predict(Xte)))
print(classification_report(yte, b.predict(Xte), target_names=['ham', 'spam']))
msg = ['win free cash prize now', 'team meeting tomorrow please review the report']
print(['spam' if p else 'ham' for p in a.predict(cv.transform(msg).toarray().astype(float))])